```
# Lab type: debug
# Course: ML402 — Reinforcement Learning
# Lesson: Dynamic Programming — Policy and Value Iteration
# Task: The code below contains 3 bugs. All run without errors but produce wrong results.
#       Find each bug, explain it in the markdown cell below it, and fix it.
```

In [ ]:
ROWS, COLS = 4, 4
GOAL = (3, 3)
PENALTIES = {(1, 1): -5, (2, 2): -5}
WALL = (1, 2)
GAMMA = 0.95
THETA = 1e-6
ACTIONS = {'N': (-1, 0), 'S': (1, 0), 'E': (0, 1), 'W': (0, -1)}

def is_valid(r, c):
    return 0 <= r < ROWS and 0 <= c < COLS and (r, c) != WALL

def step(state, action):
    if state == GOAL:
        return state, 0
    r, c = state
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc
    if not is_valid(nr, nc):
        nr, nc = r, c
    next_state = (nr, nc)
    reward = 10 if next_state == GOAL else PENALTIES.get(next_state, -1)
    return next_state, reward

def print_grid(grid_dict, label):
    print(f"\n{label}")
    for r in range(ROWS):
        row = []
        for c in range(COLS):
            val = grid_dict[(r, c)]
            row.append(f"{val:>8}" if isinstance(val, float) else f"{val:>8}")
        print("  ".join(row))

# Reference: correct synchronous value iteration
def value_iteration_ref():
    V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS)}
    sweep = 0
    while True:
        sweep += 1
        V_new = {}
        delta = 0.0
        for r in range(ROWS):
            for c in range(COLS):
                state = (r, c)
                if state == GOAL:
                    V_new[state] = 0.0
                    continue
                best = max(
                    GAMMA * V[step(state, a)[0]] + step(state, a)[1]
                    for a in ACTIONS
                )
                V_new[state] = best
                delta = max(delta, abs(V_new[state] - V[state]))
        V = V_new
        if delta < THETA:
            print(f"Reference VI converged in {sweep} sweeps")
            break
    return V

V_ref = value_iteration_ref()
print_grid({s: round(v, 2) for s, v in V_ref.items()}, "Reference optimal V")


## Bug 1: In-place value update

Value iteration below converges and prints a reasonable-looking grid, but the update rule violates the synchronous requirement of the standard Bellman sweep.

Run the cell and compare the output against the reference `V_ref` printed above.

In [ ]:
# --- BUGGY CODE (Bug 1) ---

def value_iteration_bug1():
    V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS)}
    sweep = 0
    while True:
        sweep += 1
        delta = 0.0
        for r in range(ROWS):
            for c in range(COLS):
                state = (r, c)
                if state == GOAL:
                    continue
                old_v = V[state]
                best = max(
                    GAMMA * V[step(state, a)[0]] + step(state, a)[1]
                    for a in ACTIONS
                )
                V[state] = best   # BUG: updates V in-place mid-sweep
                delta = max(delta, abs(V[state] - old_v))
        if delta < THETA:
            print(f"Bug 1 VI converged in {sweep} sweeps")
            break
    return V

V_bug1 = value_iteration_bug1()
print_grid({s: round(v, 2) for s, v in V_bug1.items()}, "Bug 1 V")
print()
print("Max absolute difference from reference:", max(abs(V_bug1[s] - V_ref[s]) for s in V_ref))


**Explain the bug:** What is a synchronous (Jacobi) update, and why does writing back to `V` during the same sweep produce a different result from the standard algorithm? In which types of environment layout does this cause the extracted policy to differ from the correct one?

*(Write your answer here.)*

In [ ]:
# Fix for Bug 1: build V_new each sweep; swap in only after the full sweep completes

def value_iteration_fix1():
    V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS)}
    sweep = 0
    while True:
        sweep += 1
        V_new = {}
        delta = 0.0
        for r in range(ROWS):
            for c in range(COLS):
                state = (r, c)
                if state == GOAL:
                    V_new[state] = 0.0
                    continue
                best = max(
                    GAMMA * V[step(state, a)[0]] + step(state, a)[1]
                    for a in ACTIONS
                )
                V_new[state] = best
                delta = max(delta, abs(V_new[state] - V[state]))
        V = V_new  # full swap after every state is processed
        if delta < THETA:
            print(f"Fix 1 VI converged in {sweep} sweeps")
            break
    return V

V_fix1 = value_iteration_fix1()
print_grid({s: round(v, 2) for s, v in V_fix1.items()}, "Fix 1 V")
print("Max absolute difference from reference:", max(abs(V_fix1[s] - V_ref[s]) for s in V_ref))


## Bug 2: Discount factor applied to the wrong term

The Bellman optimality equation is `V(s) = max_a [ R(s,a,s') + γ V(s') ]`.
The buggy implementation below applies `γ` to the immediate reward instead of the future value.

Run the cell and observe how the value grid differs from the reference.

In [ ]:
# --- BUGGY CODE (Bug 2) ---

def value_iteration_bug2():
    V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS)}
    sweep = 0
    while True:
        sweep += 1
        V_new = {}
        delta = 0.0
        for r in range(ROWS):
            for c in range(COLS):
                state = (r, c)
                if state == GOAL:
                    V_new[state] = 0.0
                    continue
                best = max(
                    V[step(state, a)[0]] + GAMMA * step(state, a)[1]  # BUG: γ on reward, not V(s')
                    for a in ACTIONS
                )
                V_new[state] = best
                delta = max(delta, abs(V_new[state] - V[state]))
        V = V_new
        if delta < THETA:
            print(f"Bug 2 VI converged in {sweep} sweeps")
            break
    return V

V_bug2 = value_iteration_bug2()
print_grid({s: round(v, 2) for s, v in V_bug2.items()}, "Bug 2 V  (γ on reward)")
print_grid({s: round(v, 2) for s, v in V_ref.items()},  "Reference V")


**Explain the bug:** The correct Bellman update discounts the *future value* V(s'), not the immediate reward. What does applying γ to the reward instead achieve mathematically? How does it distort the learned values, and what policy would be extracted from `V_bug2`?

*(Write your answer here.)*

In [ ]:
# Fix for Bug 2: γ multiplies V(s'), not the reward

def value_iteration_fix2():
    V = {(r, c): 0.0 for r in range(ROWS) for c in range(COLS)}
    sweep = 0
    while True:
        sweep += 1
        V_new = {}
        delta = 0.0
        for r in range(ROWS):
            for c in range(COLS):
                state = (r, c)
                if state == GOAL:
                    V_new[state] = 0.0
                    continue
                best = max(
                    GAMMA * V[step(state, a)[0]] + step(state, a)[1]  # correct: γ * V(s') + r
                    for a in ACTIONS
                )
                V_new[state] = best
                delta = max(delta, abs(V_new[state] - V[state]))
        V = V_new
        if delta < THETA:
            print(f"Fix 2 VI converged in {sweep} sweeps")
            break
    return V

V_fix2 = value_iteration_fix2()
print_grid({s: round(v, 2) for s, v in V_fix2.items()}, "Fix 2 V")
print("Max absolute difference from reference:", max(abs(V_fix2[s] - V_ref[s]) for s in V_ref))


## Bug 3: Policy extraction uses `min` instead of `max`

The value table below is the correct `V_ref`. The policy extraction function runs without error and produces a complete policy — but the agent following it will take the worst possible path instead of the optimal one.

In [ ]:
# --- BUGGY CODE (Bug 3) ---

def extract_policy_bug3(V):
    policy = {}
    for r in range(ROWS):
        for c in range(COLS):
            state = (r, c)
            if state == GOAL:
                policy[state] = 'G'
                continue
            worst_action = min(           # BUG: min instead of max
                ACTIONS,
                key=lambda a: GAMMA * V[step(state, a)[0]] + step(state, a)[1]
            )
            policy[state] = worst_action
    return policy

policy_bug3 = extract_policy_bug3(V_ref)
print_grid(policy_bug3, "Bug 3 policy  (should route toward GOAL — does it?)")


**Explain the bug:** The agent with `policy_bug3` starts at (0,0). Trace the first few steps using the printed grid above. Where does it go, and why? What property of the value function makes `min` produce the opposite of the optimal policy?

*(Write your answer here.)*

In [ ]:
# Fix for Bug 3: use max to select the highest-value action

def extract_policy_fix3(V):
    policy = {}
    for r in range(ROWS):
        for c in range(COLS):
            state = (r, c)
            if state == GOAL:
                policy[state] = 'G'
                continue
            best_action = max(
                ACTIONS,
                key=lambda a: GAMMA * V[step(state, a)[0]] + step(state, a)[1]
            )
            policy[state] = best_action
    return policy

policy_fix3 = extract_policy_fix3(V_ref)
print_grid(policy_fix3, "Fix 3 policy  (optimal: routes around penalties to GOAL)")


## Summary

> **For each bug, write one sentence on what went wrong and how to catch it early.**

1.
2.
3.